# Governance & Security with Unity Catalog

Access control, data masking, row-level security, and observability with Unity Catalog.

| Domain | Key Concept | Exam Keywords |
|--------|-------------|---------------|
| Access Control | GRANT / REVOKE / DENY | privilege hierarchy, principal |
| Data Masking | Column mask functions | `SET MASK`, `DROP MASK` |
| Row Security | Row filters | `SET ROW FILTER`, `DROP ROW FILTER` |
| Lineage | System tables | `system.access.audit`, `system.lineage` |
| Table Lifecycle | Managed vs External | DROP TABLE behavior |

## Setup

In [0]:
%run ../../setup/00_setup

### Configuration
Define paths to data source files used throughout this notebook.

In [0]:
# Paths to data directories (subdirectories in DATASET_PATH from 00_setup)
CUSTOMERS_PATH = f"{DATASET_PATH}/customers"
ORDERS_PATH = f"{DATASET_PATH}/orders"
PRODUCTS_PATH = f"{DATASET_PATH}/products"

# Paths to specific files
CUSTOMERS_CSV = f"{CUSTOMERS_PATH}/customers.csv"
ORDERS_JSON = f"{ORDERS_PATH}/orders_batch.json"
PRODUCTS_PARQUET = f"{PRODUCTS_PATH}/products.parquet"

## Unity Catalog Architecture

**Unity Catalog** is a unified governance solution for Databricks Lakehouse.

### Object Hierarchy:

![alt text](../../../assets/images/96129c8a7ada414b9bb6e48372164e55.png)

### Three-level namespace:
```sql
catalog.schema.table
```

Example:
```sql
main.sales.orders
dev.analytics.customer_metrics
prod.gold.daily_revenue
```

### Key Features:
- **Unified governance**: single platform for data, ML, BI
- **Fine-grained access control**: table, column, row level
- **Automatic lineage**: end-to-end data flow tracking
- **Audit logging**: who accessed what and when
- **Data discovery**: metadata search and tagging


## Access Management: GRANT / REVOKE

Unity Catalog uses **SQL-standard GRANT / REVOKE** commands to control who can access what.

### Permission hierarchy (must grant each level top-down)

```
CATALOG → SCHEMA → TABLE / VIEW / FUNCTION
```

| Permission | Grants access to |
|---|---|
| `USE CATALOG` | Opens the catalog (required first) |
| `USE SCHEMA` | Opens a schema inside the catalog |
| `SELECT` | Read data from a table / view |
| `MODIFY` | INSERT, UPDATE, DELETE on a table |
| `CREATE TABLE` | Create new tables inside a schema |
| `EXECUTE` | Call a function / UDF |
| `ALL PRIVILEGES` | All of the above at once |

Grant to **users** (`user@company.com`) or **groups** (recommended for scale).

> **Important:** Always grant `USE CATALOG` before `USE SCHEMA`, and `USE SCHEMA` before `SELECT`.


### Creating User Groups
We create user groups for permission demonstration:
- `data_engineers`: Full access to Bronze/Silver schemas
- `data_analysts`: Read-only access to Gold

In [0]:
# Verification of created schemas
schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG}").select("databaseName").collect()
schema_names = [row.databaseName for row in schemas]

**Active catalog and schema set**

We set the default working context - all subsequent operations will be executed in this catalog and schema unless a full path is specified.

In [0]:
# Verification of created schemas
spark.sql(f"SHOW SCHEMAS IN {CATALOG}").display()

## Data Preparation

Before we proceed to access management, we will load real data from the dataset/ directory that we will use in the Unity Catalog examples.

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.orders")

orders_df = spark.read.option("header", "true").option("inferSchema", "true").json(ORDERS_JSON)
orders_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.orders")

display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.orders"))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")

customers_df = spark.read.option("header", "true").option("inferSchema", "true").csv(CUSTOMERS_CSV)
customers_df.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers"))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.products")

products_df = spark.read.parquet(PRODUCTS_PARQUET)
products_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.products")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.products"))

In [0]:
# Verification of orders record count
spark.sql(f"SELECT COUNT(*) as count FROM {CATALOG}.{BRONZE_SCHEMA}.orders").display()

**Setting permissions for data-analysts**

The `data-analysts` group received:
- **USE CATALOG**: Access to catalog
- **USE SCHEMA**: Access to Silver schema 
- **SELECT**: Read data from Silver schema

**Setup:** Create groups for demonstration purposes
> **Note:** This requires account admin privileges. If you don't have them, ensure these groups exist.

* TO DO IN GUI

In [0]:
# Grant catalog access to data analysts
spark.sql(f"""
    GRANT USE CATALOG ON CATALOG {CATALOG} TO `analysts`
""")

spark.sql(f"""
    GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`
""")

spark.sql(f"""
    GRANT SELECT ON SCHEMA {CATALOG}.{SILVER_SCHEMA} TO `analysts`
""")

**Permissions for Data Analysts (Gold Layer):**

In [0]:
# GRANT for data-analysts on Gold schema
spark.sql(f"""
  GRANT USE SCHEMA ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `analysts`
""")

spark.sql(f"""
  GRANT SELECT ON SCHEMA {CATALOG}.{GOLD_SCHEMA} TO `analysts`
""")

**Table-specific access control**

Fine-grained permissions:
- **finance-team**: Access to fact_sales (revenue analysis)
- **marketing-team**: Access to customers_masked (customer insights with PII masking)

In [0]:
# GRANT EXECUTE na Functions
spark.sql(f"""
  GRANT EXECUTE ON FUNCTION {CATALOG}.{SILVER_SCHEMA}.mask_customer_id TO `analysts`
""")

spark.sql(f"""
  GRANT EXECUTE ON FUNCTION {CATALOG}.{SILVER_SCHEMA}.categorize_price TO `analysts`
""")

In [0]:
# Verify permissions on table
spark.sql(f"""
    SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
""").display()

## DENY Privilege

Unity Catalog supports **DENY** to explicitly block a permission — takes precedence over GRANT.

```sql
-- Syntax:
DENY <privilege> ON <object_type> <object_name> TO <principal>

-- Examples:
DENY SELECT ON TABLE catalog.bronze.orders TO `contractors`;
DENY MODIFY  ON SCHEMA catalog.silver      TO `data-analysts`;
DENY ALL PRIVILEGES ON CATALOG main        TO `external-users`;
```

| Command | Effect |
|---------|--------|
| `DENY SELECT` | User cannot read the table, even if GRANT SELECT exists elsewhere |
| `DENY MODIFY` | Blocks INSERT / UPDATE / DELETE / MERGE |
| `DENY ALL PRIVILEGES` | Blocks all operations on the object |

> **Exam Rule:** DENY always wins over GRANT. If a user is in two groups — one GRANTed and one DENYed — DENY takes effect.

In [0]:
# DENY takes precedence over GRANT
# Scenario: contractors group is granted USE CATALOG + USE SCHEMA by default,
# but we DENY SELECT on the sensitive orders table.

# spark.sql(f"""
#     DENY SELECT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders TO `contractors`
# """)

# Verify: SHOW GRANTS includes both GRANT and DENY entries
# spark.sql(f"SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders").display()

print("DENY privilege: blocks access even if GRANT exists on same or parent object.")

## Managed vs External Tables — Lifecycle

| Property | Managed Table | External Table |
|----------|---------------|----------------|
| Data location | UC-managed storage | User-specified external path |
| `DROP TABLE` | Deletes **both** metadata AND data files | Deletes only metadata; data files remain |
| `CREATE TABLE` syntax | No `LOCATION` clause | Requires `LOCATION 's3://...'` |
| UC ownership | Full (Unity Catalog controls) | Partial (metadata only) |
| Recommended use | Default for new tables | Legacy data, shared data |

```sql
-- Managed table (default)
CREATE TABLE catalog.schema.orders (id BIGINT, amount DOUBLE);

-- External table (data at custom path)
CREATE TABLE catalog.schema.orders_ext (id BIGINT, amount DOUBLE)
LOCATION 'abfss://container@storage.dfs.core.windows.net/orders/';
```

> **Exam Tip:** Dropping a managed table removes the data permanently.
> Dropping an external table removes only the catalog entry — the files survive.

### Native Column Masks & Row Filters (ALTER TABLE)

Beyond view-based masking, Unity Catalog supports **native column masks** and **row filters** applied directly to tables. This is the preferred approach — security is enforced at the table level, regardless of how users query the data.

| Feature | View-Based (shown above) | Native (ALTER TABLE) |
|---------|------------------------|---------------------|
| Scope | Only when querying the view | Every query on the table |
| Setup | CREATE VIEW with CASE WHEN | CREATE FUNCTION + ALTER TABLE |
| Maintenance | Must update view if logic changes | Update function only |

**Column Masks & Row Filters — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Create mask function | `CREATE FUNCTION mask_email(email STRING) RETURN CASE WHEN is_account_group_member('admins') THEN email ELSE '***' END` |
| Apply column mask | `ALTER TABLE t ALTER COLUMN email SET MASK catalog.schema.mask_email` |
| Drop column mask | `ALTER TABLE t ALTER COLUMN email DROP MASK` |
| Create row filter | `CREATE FUNCTION filter_fn(status STRING) RETURN is_account_group_member('admins') OR status = 'active'` |
| Apply row filter | `ALTER TABLE t SET ROW FILTER catalog.schema.filter_fn ON (status)` |
| Drop row filter | `ALTER TABLE t DROP ROW FILTER` |

#### Step 1: Create a Masking Function

A SQL UDF that returns the masked value. It receives the original column value and can check group membership.

In [0]:
# Create a masking function for email column
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{GOLD_SCHEMA}.mask_email_native(email STRING)
    RETURNS STRING
    RETURN CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('admins') THEN email
        ELSE CONCAT('***@', SUBSTRING_INDEX(email, '@', -1))
    END
""")

print("Masking function created: mask_email_native")

#### Step 2: Apply Mask to a Column with ALTER TABLE

`SET MASK` attaches the function directly to the column. Every query on this table will automatically apply the mask.

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.customers LIMIT 5"))

In [0]:
# Apply mask to the email column
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
    ALTER COLUMN email SET MASK {CATALOG}.{GOLD_SCHEMA}.mask_email_native
""")

# Verify — non-admins will see masked emails
display(spark.sql(f"SELECT customer_id, first_name, email FROM {CATALOG}.{BRONZE_SCHEMA}.customers LIMIT 5"))

#### Step 3: Create and Apply a Row Filter

Row filters restrict which rows are visible to different users. Like column masks, they are applied directly to the table.

In [0]:
display(spark.sql(f"SELECT customer_id, first_name, customer_segment FROM {CATALOG}.{BRONZE_SCHEMA}.customers LIMIT 10"))

In [0]:
# Create a row filter function — admins see all, others see only 'active' customers
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{BRONZE_SCHEMA}.filter_active_customers(customer_segment STRING)
    RETURNS BOOLEAN
    RETURN IF(IS_ACCOUNT_GROUP_MEMBER('admins'), true, customer_segment = 'Premium')
""")

# Apply row filter to the table
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
    SET ROW FILTER {CATALOG}.{BRONZE_SCHEMA}.filter_active_customers ON (customer_segment)
""")



In [0]:
# Verify — non-admins will only see 'active' customers
display(spark.sql(f"SELECT customer_id, first_name, customer_segment FROM {CATALOG}.{BRONZE_SCHEMA}.customers LIMIT 10"))

#### Step 4: Cleanup — Removing Masks and Filters

Use `DROP MASK` and `DROP ROW FILTER` to remove security policies from a table.

In [0]:
# Remove column mask
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
    ALTER COLUMN email DROP MASK
""")

# Remove row filter
spark.sql(f"""
    ALTER TABLE {CATALOG}.{BRONZE_SCHEMA}.customers
    DROP ROW FILTER
""")

print("Column mask and row filter removed successfully")

## Monitoring & Observability (System Tables)

Unity Catalog provides **System Tables** (`system.*`) for operational monitoring and observability.
These tables give insights into costs, job runs, pipeline health, query performance, and storage usage.

**Available System Table Categories:**

| Category | Schema | Key Tables |
|----------|--------|------------|
| **Billing** | `system.billing` | `usage`, `list_prices` |
| **Compute** | `system.compute` | `clusters`, `warehouse_events` |
| **Workflows** | `system.lakeflow` | `job_run_timeline`, `job_task_run_timeline` |
| **Pipelines** | `system.lakeflow` | `pipeline_event_log`, `pipeline_update_timeline` |
| **Queries** | `system.query` | `history` |
| **Storage** | `system.storage` | `predictive_optimization_operations_history` |
| **Access** | `system.access` | `audit`, `table_lineage`, `column_lineage` |

> **Note**: System tables require **Metastore admin** or specific `MONITOR` privileges.

**System Tables — Quick Reference**

| Table | What It Tracks |
|-------|---------------|
| `system.billing.usage` | DBU consumption by workspace, SKU, cluster, date |
| `system.query.history` | SQL query history — duration, user, warehouse, status |
| `system.compute.clusters` | Cluster events, uptime, compute type |
| `system.access.audit` | Unity Catalog access, permission changes, login events |
| `system.access.table_lineage` | Column-level data lineage across tables/views |
| `system.lakeflow.job_run_timeline` | Job run history, duration, success/failure |
| `system.lakeflow.pipeline_event_log` | Lakeflow pipeline events and data quality metrics |

## Summary

### Cost Management

| Topic | Key Concept |
|---|---|
| DBU | Billing unit — varies by compute type and SKU |
| OPTIMIZE | Fewer files → lower compute cost per query |
| System Tables | `system.billing.usage`, `system.compute.clusters` |
| Job Cluster | Always cheaper than All-purpose for production |
| Serverless | Instant start, per-query billing, zero idle cost |

### Security & Governance

| Topic | Key Concept |
|---|---|
| Unity Catalog | 3-level namespace: catalog.schema.table |
| Column Masking | `ALTER TABLE … ALTER COLUMN … SET MASK` |
| Row Filter | `ALTER TABLE … SET ROW FILTER` |
| Audit Logs | `system.access.audit` — all access events |

## Delta Sharing

**Delta Sharing** is an open protocol for securely sharing live Delta Lake data with external recipients — without copying data or requiring them to use Databricks.

| Concept | Description |
|---------|-------------|
| **Share** | A named container of schemas/tables/volumes to share |
| **Recipient** | An external party that receives access (identified by email or token) |
| **Provider** | The organization sharing data |
| **Delta Sharing Server** | Databricks Metastore acts as the server |

**Key characteristics:**
- No data duplication — recipients query live Delta data via HTTP
- Open protocol — recipients can use pandas, Spark, Power BI, or any Delta Sharing client
- Governed by Unity Catalog — uses the same GRANT model

**When to use:**
- Share data with external partners without giving them Databricks access
- Cross-cloud or cross-region data distribution
- Publishing datasets to data consumers

> **Exam note:** Delta Sharing is part of Unity Catalog governance. The exam tests whether you understand it as a _sharing_ mechanism (not a copy), and that it requires the **Metastore-level** Delta Sharing feature to be enabled.

In [0]:
%sql
-- Delta Sharing: key SQL commands (informational — requires Metastore admin)

-- Create a share
-- CREATE SHARE retailhub_share;

-- Add a table to the share
-- ALTER SHARE retailhub_share ADD TABLE retailhub_gold.customer_orders_summary;

-- Create a recipient
-- CREATE RECIPIENT external_partner;

-- Grant the share to the recipient
-- GRANT SELECT ON SHARE retailhub_share TO RECIPIENT external_partner;

-- List existing shares and recipients
SHOW SHARES;
-- SHOW RECIPIENTS;

← [09 — CI/CD & Automation](09_cicd_and_automation.ipynb) | **[README](../../../README.md)**